# Kaggle · 02 · GNN-PPO Training

**Session type: GPU (T4 or P100) — counts toward your 30 GPU hrs/week.**

**Prerequisite:** Run `Kaggle_01_preprocess.ipynb` first and publish its output as the `compiler-opt-graphs` dataset.

This notebook trains the full GNN-PPO agent across 5 random seeds and saves:
- Model checkpoints every 100 episodes
- `training_curves.csv` — reward per episode per seed (for Fig 1)
- `embeddings.pt` — GNN embeddings of all test programs (for t-SNE, Fig 4)
- `pass_selections.csv` — pass frequencies on the test set (for Fig 3)

**Estimated GPU time:** ~4–6 hours total (all 5 seeds).
**Strategy:** Run 2–3 seeds per session to stay within your weekly quota.
The notebook resumes from the last completed seed automatically.

---

In [ ]:
import subprocess, sys
def run(cmd): return subprocess.run(cmd,shell=True,capture_output=True,text=True).returncode

print("Installing LLVM 14...")
run("apt-get update -qq && apt-get install -y -qq llvm-14 clang-14")
run("update-alternatives --install /usr/bin/clang clang /usr/bin/clang-14 100")
run("update-alternatives --install /usr/bin/opt   opt   /usr/bin/opt-14   100")

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu117' if torch.cuda.is_available() else 'cpu'
print(f"\nPyTorch {TORCH_VER}, CUDA: {torch.cuda.is_available()}, tag: {CUDA_TAG}")

run("pip install -q torch_geometric")
run(f"pip install -q torch_scatter torch_sparse torch_cluster "
    f"-f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html")
run("pip install -q tqdm pandas")
print("Dependencies installed.")

## 1 · Config

In [ ]:
CFG = {
    'gnn': {
        'hidden_dim': 128, 'output_dim': 256,
        'num_layers': 3, 'dropout': 0.1,
        'use_cfg_edges': True, 'use_dfg_edges': True,
        'use_belongs_edges': True, 'use_call_edges': True,
    },
    'ppo': {
        'learning_rate': 3e-4, 'batch_size': 64,
        'buffer_size': 2048, 'n_epochs': 4,
        'gamma': 0.99, 'gae_lambda': 0.95,
        'clip_eps': 0.2, 'value_coef': 0.5,
        'entropy_coef': 0.01, 'max_grad_norm': 0.5,
    },
    'reward': {'mode': 'proxy', 'w_size': 0.5, 'w_time': 0.5},
    'passes': {
        'candidates': [
            '-mem2reg','-gvn','-licm','-loop-unroll','-inline',
            '-dse','-adce','-simplifycfg','-instcombine','-reassociate',
            '-sccp','-sroa','-early-cse','-jump-threading','-loop-rotate',
            '-loop-deletion','-loop-vectorize','-slp-vectorizer',
            '-aggressive-instcombine','-indvars',
            '-tailcallelim','-mergereturn','-constprop','-reg2mem','TERMINAL'
        ],
        'max_episode_length': 20,
    },
    'training': {
        'total_episodes': 1000,
        'val_interval': 50,
        'save_interval': 100,
        'num_seeds': 5,
    }
}

PASSES    = CFG['passes']['candidates']
N_ACTIONS = len(PASSES)
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device    : {DEVICE}")
print(f"N actions : {N_ACTIONS}")
print(f"Passes    : {PASSES}")

## 2 · Graph Builder & GNN Encoder (embedded)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, re, json, time, random, shutil, tempfile
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData, Batch
from torch_geometric.nn import HeteroConv, GATConv, global_mean_pool

# ── Opcode vocab (must match preprocessing notebook) ─────────────────────────
OPCODES=['alloca','load','store','add','sub','mul','sdiv','udiv','srem','urem',
         'fadd','fsub','fmul','fdiv','and','or','xor','shl','lshr','ashr',
         'icmp','fcmp','br','switch','ret','unreachable','call','invoke',
         'phi','select','getelementptr','bitcast','zext','sext','trunc',
         'ptrtoint','inttoptr','fpext','fptrunc','extractvalue','insertvalue','other']
OPCODE_TO_IDX={op:i for i,op in enumerate(OPCODES)}
NUM_OPCODES=len(OPCODES)

def count_instructions(ll_path):
    try:
        ct=0
        with open(ll_path,errors='replace') as f:
            for l in f:
                s=l.strip()
                if s and not any(s.startswith(p) for p in [';','!','define','declare','@','target','attributes','source_filename']) and s not in ('','{','}'):
                    ct+=1
        return max(ct,1)
    except: return 1

print(f"Opcode vocab: {NUM_OPCODES}")

In [ ]:
# ── HeteroGNN Encoder ─────────────────────────────────────────────────────────
class HeteroGNNEncoder(nn.Module):
    def __init__(self, hidden=128, out=256, layers=3, drop=0.1, edge_cfg=None):
        super().__init__()
        self.hidden=hidden; self.out=out; self.drop=drop
        self.edge_cfg=edge_cfg or {'use_cfg':True,'use_dfg':True,'use_belongs':True,'use_calls':True}
        self.proj=nn.ModuleDict({
            'func': nn.Linear(3, hidden),
            'bb':   nn.Linear(5, hidden),
            'instr':nn.Linear(NUM_OPCODES, hidden),
        })
        self.convs=nn.ModuleList()
        self.norms=nn.ModuleList()
        for _ in range(layers):
            cd={}
            if self.edge_cfg.get('use_cfg'):     cd[('bb','cfg','bb')]=GATConv(hidden,hidden,heads=4,concat=False,dropout=drop,add_self_loops=False)
            if self.edge_cfg.get('use_dfg'):     cd[('instr','dfg','instr')]=GATConv(hidden,hidden,heads=4,concat=False,dropout=drop,add_self_loops=False)
            if self.edge_cfg.get('use_belongs'): cd[('instr','belongs','bb')]=GATConv((-1,-1),hidden,heads=4,concat=False,dropout=drop,add_self_loops=False)
            if self.edge_cfg.get('use_calls'):   cd[('func','calls','func')]=GATConv(hidden,hidden,heads=4,concat=False,dropout=drop,add_self_loops=False)
            if not cd: cd[('bb','cfg','bb')]=GATConv(hidden,hidden,heads=1,concat=False,dropout=drop,add_self_loops=True)
            self.convs.append(HeteroConv(cd,aggr='sum'))
            self.norms.append(nn.ModuleDict({'func':nn.LayerNorm(hidden),'bb':nn.LayerNorm(hidden),'instr':nn.LayerNorm(hidden)}))
        self.read=nn.Sequential(nn.Linear(3*hidden,out),nn.ReLU(),nn.Linear(out,out))

    def forward(self, data):
        dev=next(self.parameters()).device
        xd={k:F.relu(self.proj[k](data[k].x.to(dev))) for k in ('func','bb','instr')
            if k in data.node_types and data[k].x.numel()>0}
        for k in ('func','bb','instr'):
            if k not in xd: xd[k]=torch.zeros(1,self.hidden,device=dev)
        ei={k:v.to(dev) for k,v in data.edge_index_dict.items()}
        for i,conv in enumerate(self.convs):
            xn=conv(xd,ei)
            xd={k:F.dropout(self.norms[i][k](F.relu(xn.get(k,xd[k]))+xd[k]),p=self.drop,training=self.training) for k in xd}
        pooled=[]
        for k in ('func','bb','instr'):
            h=xd[k]
            if hasattr(data[k],'batch') and data[k].batch is not None:
                pooled.append(global_mean_pool(h,data[k].batch.to(dev)))
            else:
                pooled.append(h.mean(0,keepdim=True))
        return self.read(torch.cat(pooled,dim=-1))

# ── Actor-Critic ──────────────────────────────────────────────────────────────
class ActorCritic(nn.Module):
    def __init__(self, enc, n_act):
        super().__init__()
        self.enc=enc; d=enc.out
        self.actor =nn.Sequential(nn.Linear(d,d//2),nn.Tanh(),nn.Linear(d//2,n_act))
        self.critic=nn.Sequential(nn.Linear(d,d//2),nn.Tanh(),nn.Linear(d//2,1))
    def forward(self,data):
        e=self.enc(data)
        return self.actor(e),self.critic(e).squeeze(-1)
    def act(self,data):
        with torch.no_grad():
            lg,v=self.forward(data)
            d=torch.distributions.Categorical(logits=lg)
            a=d.sample()
        return a.item(),d.log_prob(a).item(),v.item()
    def evaluate(self,states,actions):
        b=Batch.from_data_list(states)
        lg,vs=self.forward(b)
        d=torch.distributions.Categorical(logits=lg)
        return d.log_prob(actions),vs,d.entropy()

print("GNN encoder and ActorCritic defined.")

In [ ]:
# ── RL Environment ────────────────────────────────────────────────────────────
class LLVMEnv:
    """Single-program episodic environment. Proxy reward = -instr_count_ratio."""
    def __init__(self, graph_pt_path, passes, max_steps=20):
        saved = torch.load(graph_pt_path, map_location='cpu')
        self.graph        = saved['graph']
        self.baseline_n   = saved['baseline_instr']
        self.ll_path      = str(graph_pt_path).replace('/graphs/', '/ir/').replace('.pt', '.ll')
        self.passes       = passes
        self.max_steps    = max_steps
        self._tmpdir      = tempfile.mkdtemp(prefix='llvmenv_')
        self._cur_ll      = None
        self._step        = 0
        self._prev        = None
        self._applied     = []

    def reset(self):
        self._cleanup()
        cur = os.path.join(self._tmpdir,'cur.ll')
        if os.path.exists(self.ll_path):
            shutil.copy2(self.ll_path, cur)
        self._cur_ll = cur
        self._step=0; self._prev=None; self._applied=[]
        return self.graph  # return initial (baseline) graph

    def step(self, action):
        p = self.passes[action]
        done = False
        if p == 'TERMINAL':
            return self.graph, self._reward(), True, {'reason':'terminal','passes':self._applied}
        if p == self._prev:   # dedup
            self._step += 1
            if self._step >= self.max_steps: done=True
            return self.graph, self._reward(), done, {'skipped':True}
        # Apply pass
        with tempfile.NamedTemporaryFile(suffix='.ll', delete=False, dir=self._tmpdir) as f:
            new_ll = f.name
        r = subprocess.run(['opt', p, '-S', self._cur_ll, '-o', new_ll],
                           capture_output=True, timeout=30)
        if r.returncode == 0 and os.path.exists(new_ll):
            try: os.unlink(self._cur_ll)
            except: pass
            self._cur_ll = new_ll
            self._prev   = p
            self._applied.append(p)
        self._step += 1
        if self._step >= self.max_steps: done=True
        return self.graph, self._reward(), done, {'passes':self._applied}

    def _reward(self):
        if self._cur_ll and os.path.exists(self._cur_ll):
            cur_n = count_instructions(self._cur_ll)
            return float(-(cur_n / self.baseline_n - 1.0))
        return 0.0

    def get_metrics(self):
        cur_n = count_instructions(self._cur_ll) if self._cur_ll and os.path.exists(self._cur_ll) else self.baseline_n
        return {'instr_ratio': cur_n/self.baseline_n,
                'size_reduction': (1 - cur_n/self.baseline_n)*100,
                'passes': list(self._applied)}

    def _cleanup(self):
        if self._cur_ll and os.path.exists(self._cur_ll):
            try: os.unlink(self._cur_ll)
            except: pass

    def close(self): shutil.rmtree(self._tmpdir, ignore_errors=True)

print("LLVMEnv defined.")

In [ ]:
# ── PPO Training Loop ─────────────────────────────────────────────────────────
class RolloutBuffer:
    def __init__(self): self.clear()
    def clear(self): self.states=[]; self.actions=[]; self.rewards=[]; self.logps=[]; self.vals=[]; self.dones=[]
    def add(self,s,a,r,lp,v,d): self.states.append(s); self.actions.append(a); self.rewards.append(r); self.logps.append(lp); self.vals.append(v); self.dones.append(d)
    def size(self): return len(self.rewards)
    def gae(self, gamma=0.99, lam=0.95, last_v=0.):
        R=len(self.rewards); adv=torch.zeros(R); gae_=0.
        for t in reversed(range(R)):
            nv=last_v if t==R-1 else self.vals[t+1]
            nd=float(self.dones[t])
            delta=self.rewards[t]+gamma*nv*(1-nd)-self.vals[t]
            gae_=delta+gamma*lam*(1-nd)*gae_; adv[t]=gae_
        ret=adv+torch.tensor(self.vals)
        adv=(adv-adv.mean())/(adv.std()+1e-8)
        return ret,adv

def ppo_update(model, opt, buf, cfg_ppo, device):
    ret,adv=buf.gae(cfg_ppo['gamma'],cfg_ppo['gae_lambda'])
    ret=ret.to(device); adv=adv.to(device)
    old_lp=torch.tensor(buf.logps,device=device)
    acts=torch.tensor(buf.actions,device=device)
    total_pl=total_vl=total_el=n=0
    for _ in range(cfg_ppo['n_epochs']):
        idx=torch.randperm(buf.size())
        for st in range(0,buf.size(),cfg_ppo['batch_size']):
            b=idx[st:st+cfg_ppo['batch_size']]
            if len(b)==0: continue
            nlp,vs,ent=model.evaluate([buf.states[i] for i in b],acts[b])
            ratio=(nlp-old_lp[b]).exp()
            s1=ratio*adv[b]; s2=ratio.clamp(1-cfg_ppo['clip_eps'],1+cfg_ppo['clip_eps'])*adv[b]
            pl=-torch.min(s1,s2).mean()
            vl=F.mse_loss(vs,ret[b])
            loss=pl+cfg_ppo['value_coef']*vl-cfg_ppo['entropy_coef']*ent.mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),cfg_ppo['max_grad_norm'])
            opt.step()
            total_pl+=pl.item(); total_vl+=vl.item(); total_el+=ent.mean().item(); n+=1
    return {'policy_loss':total_pl/max(n,1),'value_loss':total_vl/max(n,1),'entropy':total_el/max(n,1)}

print("PPO components defined.")

## 3 · Load Dataset

In [ ]:
GRAPH_ROOT = Path('/kaggle/input/compiler-opt-graphs')
OUT_DIR    = Path('/kaggle/working/results')
CKPT_DIR   = Path('/kaggle/working/checkpoints')
for d in (OUT_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)

def load_split(split):
    pts = sorted((GRAPH_ROOT/split).glob('*.pt'))
    print(f"  {split}: {len(pts)} graphs")
    return pts

train_pts = load_split('train')
val_pts   = load_split('val')
test_pts  = load_split('test')

# Verify one graph loads correctly
sample = torch.load(train_pts[0])
g = sample['graph']
print(f"\nSample graph:")
print(f"  funcs={g['func'].x.shape[0]}, bbs={g['bb'].x.shape[0]}, instrs={g['instr'].x.shape[0]}")
print(f"  baseline_instr={sample['baseline_instr']}")

## 4 · Training Loop (5 Seeds)

In [ ]:
ALL_CURVES  = []  # accumulated across seeds
TOTAL_EPISODES = CFG['training']['total_episodes']
VAL_INTERVAL   = CFG['training']['val_interval']
SAVE_INTERVAL  = CFG['training']['save_interval']
BUFFER_SIZE    = CFG['ppo']['buffer_size']

for SEED in range(CFG['training']['num_seeds']):
    # ── Check if this seed already done ──────────────────────────────────────
    done_flag = CKPT_DIR / f'seed_{SEED}_done.flag'
    if done_flag.exists():
        print(f"Seed {SEED}: already completed — skipping")
        seed_csv = OUT_DIR / f'curves_seed_{SEED}.csv'
        if seed_csv.exists():
            ALL_CURVES.append(pd.read_csv(seed_csv))
        continue

    print(f"\n{'='*60}")
    print(f"SEED {SEED} / {CFG['training']['num_seeds']-1}")
    print(f"{'='*60}")

    # Set all seeds
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    # Build model
    edge_cfg = {
        'use_cfg':     CFG['gnn']['use_cfg_edges'],
        'use_dfg':     CFG['gnn']['use_dfg_edges'],
        'use_belongs': CFG['gnn']['use_belongs_edges'],
        'use_calls':   CFG['gnn']['use_call_edges'],
    }
    enc   = HeteroGNNEncoder(CFG['gnn']['hidden_dim'], CFG['gnn']['output_dim'],
                              CFG['gnn']['num_layers'], CFG['gnn']['dropout'], edge_cfg)
    model = ActorCritic(enc, N_ACTIONS).to(DEVICE)
    optim = torch.optim.Adam(model.parameters(), lr=CFG['ppo']['learning_rate'])
    buf   = RolloutBuffer()

    seed_curves = []
    rng = random.Random(SEED)

    t_start = time.time()

    for ep in range(TOTAL_EPISODES):
        # Sample random training program
        pt = rng.choice(train_pts)
        env = LLVMEnv(pt, PASSES, CFG['passes']['max_episode_length'])
        state = env.reset()
        ep_reward = 0.0
        done = False

        while not done:
            action, logp, val = model.act(state)
            next_state, reward, done, info = env.step(action)
            buf.add(state, action, reward, logp, val, done)
            ep_reward += reward
            state = next_state

        env.close()

        # PPO update when buffer is full
        if buf.size() >= BUFFER_SIZE:
            losses = ppo_update(model, optim, buf, CFG['ppo'], DEVICE)
            buf.clear()

        seed_curves.append({'episode': ep, 'mean_reward': ep_reward,
                            'seed': SEED, 'method': 'GNN-PPO (ours)'})

        # Validation
        if (ep + 1) % VAL_INTERVAL == 0:
            val_rewards = []
            model.eval()
            for vpt in val_pts[:8]:  # quick val on subset
                venv = LLVMEnv(vpt, PASSES, CFG['passes']['max_episode_length'])
                vs = venv.reset(); vr = 0.; vd = False
                while not vd:
                    va, _, _ = model.act(vs)
                    vs, vr_step, vd, _ = venv.step(va)
                    vr += vr_step
                val_rewards.append(vr); venv.close()
            model.train()
            elapsed = (time.time() - t_start) / 3600
            print(f"  ep {ep+1:4d}/{TOTAL_EPISODES} | train_r={ep_reward:.3f} | "
                  f"val_r={np.mean(val_rewards):.3f} | {elapsed:.2f}h")

        # Save checkpoint
        if (ep + 1) % SAVE_INTERVAL == 0:
            ckpt_path = CKPT_DIR / f'gnn_ppo_seed{SEED}_ep{ep+1}.pt'
            torch.save({'model': model.state_dict(), 'optim': optim.state_dict(),
                        'episode': ep+1, 'seed': SEED, 'curves': seed_curves}, ckpt_path)

    # Save seed curves
    curves_df = pd.DataFrame(seed_curves)
    curves_df.to_csv(OUT_DIR / f'curves_seed_{SEED}.csv', index=False)
    ALL_CURVES.append(curves_df)
    done_flag.touch()
    print(f"Seed {SEED} done. Total time: {(time.time()-t_start)/3600:.2f}h")

# Combine and save all curves
all_curves_df = pd.concat(ALL_CURVES, ignore_index=True)
all_curves_df.to_csv(OUT_DIR / 'training_curves.csv', index=False)
print(f"\nAll seeds done. training_curves.csv saved ({len(all_curves_df)} rows).")

## 5 · Extract Test-Set Embeddings & Pass Selections

In [ ]:
# Load the last saved model (seed 0, last checkpoint)
ckpts = sorted(CKPT_DIR.glob('gnn_ppo_seed0_ep*.pt'))
if ckpts:
    ckpt  = torch.load(ckpts[-1], map_location=DEVICE)
    edge_cfg = {'use_cfg':True,'use_dfg':True,'use_belongs':True,'use_calls':True}
    enc   = HeteroGNNEncoder(CFG['gnn']['hidden_dim'],CFG['gnn']['output_dim'],
                              CFG['gnn']['num_layers'],CFG['gnn']['dropout'],edge_cfg)
    model = ActorCritic(enc, N_ACTIONS).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print(f"Loaded checkpoint: {ckpts[-1].name}")

    embeddings_list = []
    labels_list     = []
    pass_rows       = []
    test_metrics    = []

    for pt in tqdm(test_pts, desc='Test set evaluation'):
        saved = torch.load(pt, map_location='cpu')
        g     = saved['graph']

        # Extract embedding
        with torch.no_grad():
            emb = enc(g).squeeze(0).cpu().numpy()
        embeddings_list.append(emb)
        labels_list.append(0)  # label by problem class if available

        # Run greedy episode + collect pass frequencies
        env  = LLVMEnv(pt, PASSES, CFG['passes']['max_episode_length'])
        state = env.reset()
        done = False; pass_counts = defaultdict(int)
        while not done:
            a, _, _ = model.act(state)
            pass_counts[PASSES[a]] += 1
            state, _, done, _ = env.step(a)
        metrics = env.get_metrics()
        test_metrics.append({'program': pt.stem, **metrics})
        for p, cnt in pass_counts.items():
            pass_rows.append({'program': pt.stem, 'pass': p, 'frequency': cnt})
        env.close()

    # Save embeddings
    torch.save({'embeddings': torch.tensor(np.array(embeddings_list)),
                'labels':     torch.tensor(labels_list)},
               OUT_DIR / 'embeddings.pt')

    # Save pass selections
    pd.DataFrame(pass_rows).to_csv(OUT_DIR / 'pass_selections.csv', index=False)

    # Save per-program test results
    pd.DataFrame(test_metrics).to_csv(OUT_DIR / 'test_program_results_gnn_ppo.csv', index=False)

    print(f"\nSaved: embeddings.pt, pass_selections.csv, test_program_results_gnn_ppo.csv")
    print(f"Mean size reduction: {np.mean([m['size_reduction'] for m in test_metrics]):.1f}%")
else:
    print("No checkpoints found — training may not have completed.")

In [ ]:
# Final output listing
print("Output files:")
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file():
        print(f"  {f.relative_to(OUT_DIR)}  ({f.stat().st_size//1024} KB)")
print("\nDownload these files and place them in results/ on your M1.")
print("Then run M1_03_visualisations.ipynb to generate all paper figures.")